# 01 · Bronze — ingest raw delivery truck trips
Lands the raw Excel export from `lh_bronze/Files/raw_data` into the Delta table
**`lh_bronze.dbo.delivery_trips_raw`**, exactly as received (every value kept as a string),
plus lineage columns: `_source_file`, `_batch_id`, `_ingestion_timestamp`, `_ingestion_date`.

* Append-only: every run is a new batch, so Bronze keeps a full audit trail.
* The file is read with pandas/openpyxl because the auto-generated `dbo.raw_data`
  shortcut table loses most Excel datetime values.

In [ ]:
source_folder = "Files/raw_data"
target_table = "dbo.delivery_trips_raw"

In [ ]:
import re
import uuid
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

# Bronze is write-optimised: skip V-Order, let Fabric optimise file sizes on write.
spark.conf.set("spark.sql.parquet.vorder.default", "false")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

batch_id = str(uuid.uuid4())
ingested_at = datetime.now(timezone.utc)
files = [f for f in notebookutils.fs.ls(source_folder) if f.name.lower().endswith((".xlsx", ".csv"))]
assert files, f"No .xlsx/.csv files found in {source_folder}"
print(f"Batch {batch_id}: {[f.name for f in files]}")

In [ ]:
def delta_safe(name: str) -> str:
    """Delta column names can't contain spaces or ,;{}()=/ - replace them with underscores."""
    return re.sub(r"_+", "_", re.sub(r"[^0-9A-Za-z_]", "_", name.strip())).strip("_")


frames = []
for f in files:
    local_path = f"/lakehouse/default/{source_folder}/{f.name}"
    if f.name.lower().endswith(".xlsx"):
        pdf = pd.read_excel(local_path, dtype=str)
    else:
        pdf = pd.read_csv(local_path, dtype=str, encoding="utf-8-sig")
    pdf.columns = [delta_safe(c) for c in pdf.columns]
    pdf = pdf.astype(object).where(pdf.notna(), None)
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    frames.append(
        spark.createDataFrame(pdf, schema=schema)
        .withColumn("_source_file", F.lit(f.path))
    )
    print(f"{f.name}: {len(pdf):,} rows x {len(pdf.columns)} columns")

raw_df = frames[0]
for extra in frames[1:]:
    raw_df = raw_df.unionByName(extra, allowMissingColumns=True)

bronze_df = (
    raw_df
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_ingestion_timestamp", F.lit(ingested_at).cast("timestamp"))
    .withColumn("_ingestion_date", F.to_date("_ingestion_timestamp"))
)

In [ ]:
(bronze_df.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("_ingestion_date")
    .saveAsTable(target_table))

## Validate the batch

In [ ]:
expected = raw_df.count()
landed = spark.table(target_table).filter(F.col("_batch_id") == batch_id).count()
batches = spark.table(target_table).select("_batch_id").distinct().count()
print(f"Rows in source: {expected:,} | rows landed: {landed:,} | batches in table: {batches}")
assert landed == expected and landed > 0, "Bronze row count mismatch"

notebookutils.notebook.exit(f'{{"batch_id": "{batch_id}", "rows": {landed}}}')